In [ ]:
import pandas as pd
from functools import partial

dataset_root = "/home/xzhao/workspace/GYB_self-ensemble/datasets"
# dataset_root = "/home/y-guo/self-ensemble"

def get_ppl_filename(
        model_name, dataset_name, logits_ensemble_method, 
        is_baseline, repeat_paras, ensemble_method, 
        ensemble_layer, ensemble_alpha, token_mode, 
        multilayer, num_fewshots, num_paraphrases):
    
    if is_baseline:
        dump_file = f"{dataset_root}/{dataset_name}_paraphrase/{model_name}/{dataset_name}paraphrase.ppl.baseline."
    else:
        dump_file = f"{dataset_root}/{dataset_name}_paraphrase/{model_name}/{dataset_name}paraphrase.ppl.logits.{logits_ensemble_method}."
        if ensemble_method == "layer_output_avg":
            dump_file += f"avglayer.layer{ensemble_layer}.alpha{int(ensemble_alpha*100)}.token-{token_mode}."
        elif ensemble_method == "ffn_activation_max":
            dump_file += f"maxffn.layer{ensemble_layer}.alpha{int(ensemble_alpha*100)}.token-{token_mode}."
        if multilayer:
            dump_file += "multilayer."
    if num_fewshots != 5:
        dump_file += f"{num_fewshots}fshots."
    dump_file += f"{num_paraphrases}paras.feather"
    return dump_file

In [ ]:
num_fewshots = 0
num_paraphrases = 5

get_ppl_filename_partial = partial(
    get_ppl_filename,
    logits_ensemble_method="avg",
    num_fewshots=num_fewshots,
    num_paraphrases=num_paraphrases,
    multilayer=True,
    repeat_paras=False,
    ensemble_method="layer_output_avg", 
    ensemble_alpha=1, 
    token_mode="last")

In [3]:
from notebooks._utils import calculate_accuracy, get_layers

# model_name = "llama3.2_3b"
for model_name in ["llama3.2_3b", "qwen2.5_3b", "qwen3_4b", "pythia_2.8b", "qwen3_30b"]:
    print(f"\n================ Evaluating model: {model_name} ================ ")
    for dataset in ["commonsense", "mmlu", "logiqa"]: 
    # for dataset in ["commonsense"]: 
        print(f"\n------ Evaluating dataset: {dataset} ------")
        basefn = get_ppl_filename_partial(
            model_name=model_name,
            dataset_name=dataset,
            is_baseline=True,
            ensemble_layer=get_layers(model_name))
        
        ensemblefn = get_ppl_filename_partial(
            model_name=model_name,
            dataset_name=dataset,
            is_baseline=False,
            ensemble_layer=get_layers(model_name))
        
        try:
            basedf = pd.read_feather(basefn)
            calculate_accuracy(basedf, label="Baseline", is_multichoice=True)
        except FileNotFoundError:
            print(f"Baseline file not found: {basefn}")

        try:
            ensembledf = pd.read_feather(ensemblefn)
            calculate_accuracy(ensembledf, label="Ensemble", is_multichoice=True)
        except FileNotFoundError:
            print(f"Ensemble file not found: {ensemblefn}")


================ Evaluating model: llama3.2_3b ================ 

------ Evaluating dataset: commonsense ------
Multichoice Acc: 0.4740 ==> 🏷️ Baseline
Ensemble file not found: /home/y-guo/self-ensemble/commonsense_paraphrase/llama3.2_3b/commonsenseparaphrase.ppl.logits.avg.avglayer.layer21.alpha100.token-last.multilayer.0fshots.3paras.feather

------ Evaluating dataset: mmlu ------
Multichoice Acc: 0.4000 ==> 🏷️ Baseline
Ensemble file not found: /home/y-guo/self-ensemble/mmlu_paraphrase/llama3.2_3b/mmluparaphrase.ppl.logits.avg.avglayer.layer21.alpha100.token-last.multilayer.0fshots.3paras.feather

------ Evaluating dataset: logiqa ------
Baseline file not found: /home/y-guo/self-ensemble/logiqa_paraphrase/llama3.2_3b/logiqaparaphrase.ppl.baseline.0fshots.3paras.feather
Ensemble file not found: /home/y-guo/self-ensemble/logiqa_paraphrase/llama3.2_3b/logiqaparaphrase.ppl.logits.avg.avglayer.layer21.alpha100.token-last.multilayer.0fshots.3paras.feather

================ Evaluating mode

In [24]:
import os
os.path.exists(basefn)

False

In [25]:
basedf

,uuid,answers,prediction,generation,correctness,paraphrases,ppls,best_choice_idx,choices_label,choices_text,answer_label
0,0039e607daadf9c5727e5c2757301628,Injecting a hormonal hormone into the voles' b...,B,B,None,[University of Pavia research in Italy shows M...,"[116.58341217041016, 36.201656341552734, 228.5...",Injecting a hormonal hormone into the voles' b...,"[A, B, C, D]",[Stimulation of oxytocin only lasts about one ...,B
1,0039e607daadf9c5727e5c2757301628,Injecting a hormonal hormone into the voles' b...,B,B,None,[University of Pavia research in Italy shows M...,"[134.37600708007812, 40.55244064331055, 207.94...",Injecting a hormonal hormone into the voles' b...,"[A, B, C, D]",[Stimulation of oxytocin only lasts about one ...,B
2,0039e607daadf9c5727e5c2757301628,Injecting a hormonal hormone into the voles' b...,B,B,None,[University of Pavia research in Italy shows M...,"[106.19190979003906, 41.4101676940918, 236.216...",Injecting a hormonal hormone into the voles' b...,"[A, B, C, D]",[Stimulation of oxytocin only lasts about one ...,B
3,00704c973b872ceb25b259970350e234,"Zhao introduced 200,000 yuan in an online vent...",D,D,None,[Equity crowdfunding refers to the activity of...,"[40.990352630615234, 41.7598762512207, 19.6098...","Zhao introduced 200,000 yuan in an online vent...","[A, B, C, D]","[Yang, who is a business, has invested in a ne...",D
4,00704c973b872ceb25b259970350e234,"Zhao introduced 200,000 yuan in an online vent...",D,D,None,[Equity crowdfunding refers to the activity of...,"[48.70159149169922, 40.86629867553711, 18.4866...","Zhao introduced 200,000 yuan in an online vent...","[A, B, C, D]","[Yang, who is a business, has invested in a ne...",D
...,...,...,...,...,...,...,...,...,...,...,...
2995,ff94d70d05abafa9fe49ca762a19bf94,Obtaining a teacher qualification certificate ...,B,B,None,[University graduates who have not obtained a ...,"[19.397254943847656, 6.137147903442383, 9.4141...",Obtaining a teacher qualification certificate ...,"[A, B, C, D]",[Everyone who has obtained the teacher qualifi...,B
2996,ff94d70d05abafa9fe49ca762a19bf94,Obtaining a teacher qualification certificate ...,B,B,None,[University graduates who have not obtained a ...,"[29.305360794067383, 7.648821830749512, 9.6440...",Obtaining a teacher qualification certificate ...,"[A, B, C, D]",[Everyone who has obtained the teacher qualifi...,B
2997,ffd575d23c71ce1822d3a282824ff27a,"Xiao Li came to work happily, seeing that his ...",B,B,None,[Second-hand stress refers to the situation in...,"[25.3614444732666, 15.196998596191406, 36.8142...","As soon as Xiao Zhang entered the office, he f...","[A, B, C, D]",[Xiao Wang gets up in the morning and finds th...,D
2998,ffd575d23c71ce1822d3a282824ff27a,"Xiao Li came to work happily, seeing that his ...",B,B,None,[Second-hand stress refers to the situation in...,"[21.91436767578125, 15.44619083404541, 32.3426...","As soon as Xiao Zhang entered the office, he f...","[A, B, C, D]",[Xiao Wang gets up in the morning and finds th...,D
